# Solutions — Pivot and Unpivot (Medium 13)

**Datasets:** `sales_transactions`, `tpch.orders`, `wanderbricks.bookings`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
orders       = spark.table("samples.tpch.orders")
bookings     = spark.table("samples.wanderbricks.bookings")
properties   = spark.table("samples.wanderbricks.properties")

## Solution 1 — Pivot Payment Methods to Columns

In [ ]:
# Why: pivot turns paymentMethod values into separate columns; fill null with 0
result_1 = (
    transactions
    .groupBy("product")
    .pivot("paymentMethod")
    .agg(F.round(F.sum("totalPrice"), 2))
    .fillna(0)
    .orderBy("product")
)

## Solution 2 — Pivot Order Status by Year

In [ ]:
result_2 = (
    orders
    .withColumn("year", F.year("o_orderdate"))
    .groupBy("year")
    .pivot("o_orderstatus")
    .agg(F.count("o_orderkey"))
    .fillna(0)
    .orderBy("year")
)

## Solution 3 — Pivot Booking Status by Property Type

In [ ]:
result_3 = (
    bookings
    .join(properties, on="property_id")
    .groupBy("property_type")
    .pivot("status")
    .agg(F.count("booking_id"))
    .fillna(0)
    .orderBy("property_type")
)

## Solution 4 — Unpivot Payment Method Revenue (stack)

In [ ]:
# Why: stack() is PySpark's unpivot; column count must match the field list
pivoted = (
    transactions
    .groupBy("product")
    .pivot("paymentMethod")
    .agg(F.round(F.sum("totalPrice"), 2))
    .fillna(0)
)
payment_methods = [c for c in pivoted.columns if c != "product"]
stack_expr = f"stack({len(payment_methods)}, " + ", ".join(
    [f"'{m}', `{m}`" for m in payment_methods]) + ") as (payment_method, total_revenue)"
result_4 = (
    pivoted
    .select("product", F.expr(stack_expr))
    .filter(F.col("total_revenue") > 0)
    .orderBy("product","payment_method")
)

## Solution 5 — Crosstab Product × Payment Method

In [ ]:
# Why: crosstab creates a frequency matrix; counts instead of revenue
result_5 = transactions.crosstab("product","paymentMethod")

## Solution 6 — Multi-Level Pivot: Revenue AND Count per Method

In [ ]:
# Why: using a dict/map in agg gives multiple aggregations in one pivot
result_6 = (
    transactions
    .groupBy("product")
    .pivot("paymentMethod")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("revenue"),
        F.count(F.lit(1)).alias("count")
    )
    .fillna(0)
    .orderBy("product")
)

## Solution 7 — Monthly Revenue Pivot (Month × Product)

In [ ]:
result_7 = (
    transactions
    .withColumn("month_label", F.date_format("dateTime", "yyyy-MM"))
    .groupBy("month_label")
    .pivot("product")
    .agg(F.round(F.sum("totalPrice"), 2))
    .fillna(0)
    .orderBy("month_label")
)